# 2025/26 CSC 4792: Data Mining and Warehousing
## Mini-Project: Municipal Open Data Extraction, Cleaning, and Standardization
### Target Municipal Authority: **Ndola City Council (Republic of Zambia)**
**Project Team:** #2  
**Dataset Identifier:** `db-unza26-csc4792-ndola_city_council`  
**Delimitation / Output Specification:** Pipe-delimited CSV (`sep='|'`)

---

### Executive Summary & Methodology
This notebook documents the end-to-end data engineering lifecycle for curating a municipal open dataset for the **Ndola City Council** in Zambia. The data reflects statutory mandates under the **Local Government Act No. 2 of 2019** (and subsequent amendments), including:
1. **Constituency Development Fund (CDF)**: Community projects, skills development bursaries, secondary school boarding bursaries, and youth/women empowerment loans/grants across all 4 constituencies (*Ndola Central, Bwana M’kubwa, Chifubu, Kabushi*).
2. **Municipal Financial & Budgetary Governance**: Output-Based Budgeting (OBB), Local Government Equalisation Fund (LGEF) transfers, local taxes/rates, and audited financial statements.
3. **Institutional Governance & Committee Resolutions**: Resolutions from Ordinary Council Meetings, Planning, Audit, Engineering Services, and District Development Coordinating Committee (DDCC) sessions.


## 1. Environment Setup & Dependency Initialization
We initialize our extraction, transformation, and tabular cleaning dependencies (`pandas`, `pdfplumber`, `re`, `json`, `os`, `glob`).

In [ ]:
import os
import re
import glob
import json
import pdfplumber
import pandas as pd
from tabulate import tabulate

BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), "..")) if "notebooks" in os.getcwd() else os.getcwd()
SOURCE_DIR = os.path.join(BASE_DIR, "source_files")
TABULAR_DIR = os.path.join(BASE_DIR, "tabular")
SCRIPTS_DIR = os.path.join(BASE_DIR, "scripts")

os.makedirs(TABULAR_DIR, exist_ok=True)
print(f"Working Base Directory: {BASE_DIR}")
print(f"Source Files Directory: {SOURCE_DIR}")
print(f"Tabular Output Directory: {TABULAR_DIR}")


## 2. Ingestion Audit: Downloaded Municipal Source Artifacts
Audit the raw files collected from `https://www.ndolacouncil.gov.zm` stored in `/source_files/`.

In [ ]:
raw_files = sorted(glob.glob(os.path.join(SOURCE_DIR, "*")))
print(f"Total Source Artifacts Ingested: {len(raw_files)}")
audit_summary = []
for p in raw_files:
    fname = os.path.basename(p)
    size_kb = os.path.getsize(p) / 1024
    ext = fname.split(".")[-1].upper()
    audit_summary.append({"Filename": fname, "Type": ext, "Size_KB": round(size_kb, 2)})

audit_df = pd.DataFrame(audit_summary)
print(tabulate(audit_df.head(15), headers="keys", tablefmt="psql"))


## 3. Data Cleaning & Transformation Helpers
We implement robust string cleaning, currency standardizers (converting ZMW amounts with commas and currency symbols into clean floating-point values), and delimiter integrity safeguards (replacing accidental internal pipes).

In [ ]:
def clean_text(text):
    if text is None or pd.isna(text):
        return ""
    # Normalize whitespaces and replace pipe characters to prevent CSV delimiter corruption
    cleaned = re.sub(r"\s+", " ", str(text)).replace("|", "/").strip()
    return cleaned

def clean_amount(val):
    if not val or pd.isna(val):
        return 0.0
    val_str = str(val).replace("ZMW", "").replace("K", "").replace(",", "").strip()
    match = re.search(r"[-+]?\d*\.\d+|\d+", val_str)
    if match:
        try:
            return float(match.group(0))
        except ValueError:
            return 0.0
    return 0.0


## 4. Extraction & Cleaning: CDF Community Infrastructure Projects (2024–2025)
Extracting approved and disapproved community projects across Ndola District (*Ndola Central, Bwana M’kubwa, Chifubu, Kabushi*).

In [ ]:
cdf_projects = []

# Process 2025 Approved Projects
path_2025_app = os.path.join(SOURCE_DIR, "2025-List-of-approved-Community-Projects-Ndola-District-1.pdf")
if os.path.exists(path_2025_app):
    with pdfplumber.open(path_2025_app) as pdf:
        for page in pdf.pages:
            for tbl in page.extract_tables():
                for row in tbl:
                    row_c = [clean_text(c) for c in row]
                    if len(row_c) >= 10 and re.match(r"^\d+$", row_c[0]):
                        cdf_projects.append({
                            "project_name": row_c[1],
                            "project_description": row_c[2],
                            "sector": row_c[3].upper(),
                            "project_type": row_c[4].upper(),
                            "constituency": row_c[5].upper().replace("BWANA MKUBW", "BWANA MKUBWA"),
                            "ward": row_c[6].upper(),
                            "zone": row_c[7].upper(),
                            "location": row_c[8],
                            "amount_allocated_zmw": clean_amount(row_c[9]),
                            "approval_status": "APPROVED",
                            "fiscal_year": 2025,
                            "justification_notes": row_c[10] if len(row_c) > 10 else "Approved by Ministry/Council"
                        })

# Process 2024 Approved Projects
path_2024_app = os.path.join(SOURCE_DIR, "APPROVED-2024-CDF-COMMUNITY-PROJECTS-1.pdf")
if os.path.exists(path_2024_app):
    with pdfplumber.open(path_2024_app) as pdf:
        for page in pdf.pages:
            for tbl in page.extract_tables():
                for row in tbl:
                    row_c = [clean_text(c) for c in row]
                    if len(row_c) >= 9 and re.match(r"^\d+$", row_c[0]):
                        cdf_projects.append({
                            "project_name": row_c[1],
                            "project_description": row_c[2],
                            "sector": row_c[3].upper(),
                            "project_type": row_c[4].upper(),
                            "constituency": row_c[5].upper().replace("BWANA MKUBW", "BWANA MKUBWA"),
                            "ward": row_c[6].upper(),
                            "zone": row_c[7].upper(),
                            "location": row_c[6],
                            "amount_allocated_zmw": clean_amount(row_c[8]),
                            "approval_status": "APPROVED",
                            "fiscal_year": 2024,
                            "justification_notes": row_c[9] if len(row_c) > 9 else "Approved 2024"
                        })

# Process Disapproved Projects
path_2025_dis = os.path.join(SOURCE_DIR, "LIist-of-not-approved-CDF-2025-Community-Projects-Ndola-District-1.pdf")
if os.path.exists(path_2025_dis):
    with pdfplumber.open(path_2025_dis) as pdf:
        for page in pdf.pages:
            for tbl in page.extract_tables():
                for row in tbl:
                    row_c = [clean_text(c) for c in row]
                    if len(row_c) >= 11:
                        sn = row_c[1] if row_c[0] == "" else row_c[0]
                        if re.match(r"^\d+$", sn):
                            offset = 1 if row_c[0] == "" else 0
                            cdf_projects.append({
                                "project_name": row_c[offset + 1],
                                "project_description": row_c[offset + 2],
                                "sector": row_c[offset + 3].upper(),
                                "project_type": row_c[offset + 4].upper(),
                                "constituency": row_c[offset + 5].upper().replace("BWANA MKUBW", "BWANA MKUBWA"),
                                "ward": row_c[offset + 6].upper(),
                                "zone": row_c[offset + 7].upper(),
                                "location": row_c[offset + 8],
                                "amount_allocated_zmw": 0.0,
                                "approval_status": "DISAPPROVED",
                                "fiscal_year": 2025,
                                "justification_notes": row_c[offset + 10] if len(row_c) > (offset + 10) else row_c[offset + 9]
                            })

df_projects = pd.DataFrame(cdf_projects)
df_projects.insert(0, "project_id", [f"NCC-CDF-PRJ-{i+1:04d}" for i in range(len(df_projects))])
print(f"Total CDF Community Projects Extracted: {len(df_projects)}")
print(tabulate(df_projects.head(), headers="keys", tablefmt="psql"))


## 5. Extraction & Cleaning: CDF Skills Development Bursaries (2025)
Extracting beneficiary records across all 4 constituencies in Ndola District.

In [ ]:
skills_records = []
skills_files = [
    ("Ndola Central", "01-LiST-OF-APPROVED-CDF-2025-SKILLS-DEVELOPMENT-Ndola-Central-Constituency.pdf"),
    ("Bwana Mkubwa", "LIST-OF-APPROVED-CDF-2025-SKILLS-DEVELOPMENT-Bwana-Mkubwa_Constituency.pdf"),
    ("Chifubu", "LIST-OF-APPROVED-CDF-2025-SKILLS-DEVELOPMENT-Chifubu-constituency.pdf"),
    ("Kabushi", "LIST-OF-APPROVED-CDF-2025-SKILS-DEVELOPMENT-BENEFICARIES-KABUSHI-CONSTITUENCY.pdf"),
]

for const_name, fname in skills_files:
    fpath = os.path.join(SOURCE_DIR, fname)
    if not os.path.exists(fpath):
        continue
    with pdfplumber.open(fpath) as pdf:
        for page in pdf.pages:
            text = page.extract_text() or ""
            for line in text.split("\n"):
                m = re.match(r"^(\d+)\s+([A-Z\s\'-]+?)\s+(\d{6}/\d{2}/\d|\d{9})\s+(\d{9,10})\s+([A-Z\s]+?)\s+([MF])\s+(\d{1,2})\s+(.+?)\s+([\d,]+\.\d{2}|[\d,]+)$", line.strip())
                if m:
                    sn, name, nrc, contact, ward, gender, age, middle_blob, fees = m.groups()
                    skills_records.append({
                        "constituency": const_name.upper(),
                        "student_name": clean_text(name),
                        "nrc_no": clean_text(nrc),
                        "contact_no": clean_text(contact),
                        "ward": clean_text(ward),
                        "gender": gender.upper(),
                        "age": int(age) if age.isdigit() else None,
                        "institution_and_course": clean_text(middle_blob),
                        "annual_fees_zmw": clean_amount(fees),
                        "academic_year": 2025,
                        "approval_status": "APPROVED"
                    })
                else:
                    m2 = re.match(r"^(\d+)\s+([A-Z\s\'-]+?)\s+(\d{6}/\d{2}/\d|\d{9})\s+(.+?)\s+([\d,]+\.\d{2}|[\d,]+)$", line.strip())
                    if m2:
                        sn, name, nrc, middle_blob, fees = m2.groups()
                        g_match = re.search(r"\b([MF])\b\s+(\d{2})\b", middle_blob)
                        gender = g_match.group(1) if g_match else "U"
                        age = int(g_match.group(2)) if g_match else None
                        skills_records.append({
                            "constituency": const_name.upper(),
                            "student_name": clean_text(name),
                            "nrc_no": clean_text(nrc),
                            "contact_no": "",
                            "ward": "",
                            "gender": gender,
                            "age": age,
                            "institution_and_course": clean_text(middle_blob),
                            "annual_fees_zmw": clean_amount(fees),
                            "academic_year": 2025,
                            "approval_status": "APPROVED"
                        })

df_skills = pd.DataFrame(skills_records)
df_skills.drop_duplicates(subset=["constituency", "student_name", "nrc_no"], inplace=True)
df_skills.insert(0, "bursary_id", [f"NCC-CDF-SKL-{i+1:05d}" for i in range(len(df_skills))])
print(f"Total Skills Development Bursaries Extracted: {len(df_skills)}")
print(tabulate(df_skills.head(), headers="keys", tablefmt="psql"))


## 6. Extraction & Cleaning: CDF Secondary Boarding Bursaries (2025)

In [ ]:
sec_records = []
sec_files = [
    ("Ndola Central", "LIST-OF-APPROVED-CDF-2025-SECONDARY-BOARDING-Ndola-Central.pdf"),
    ("Bwana Mkubwa", "LIST-OF-APPROVED-CDF-2025-SECONDARY-BOARDING-Bwana-Mkubwa_Constituency.pdf"),
    ("Chifubu", "LIST-OF-APPROVED-CDF-2025-SECONDARY-BOARDING-CHIFUBU-CONSTITUENCY.pdf"),
    ("Kabushi", "LIST-OF-APPROVED-CDF-2025-SECONDARY-BOARDING-KABUSHI-Constituency.pdf"),
]

for const_name, fname in sec_files:
    fpath = os.path.join(SOURCE_DIR, fname)
    if not os.path.exists(fpath):
        continue
    with pdfplumber.open(fpath) as pdf:
        for page in pdf.pages:
            tables = page.extract_tables()
            if tables:
                for tbl in tables:
                    for row in tbl:
                        row_c = [clean_text(c) for c in row if c is not None]
                        if len(row_c) >= 7 and re.match(r"^\d+$", row_c[0]):
                            sec_records.append({
                                "constituency": const_name.upper(),
                                "pupil_name": row_c[1],
                                "ward": row_c[2] if len(row_c) > 2 else "",
                                "gender": row_c[3] if len(row_c) > 3 else "",
                                "age": int(row_c[4]) if len(row_c) > 4 and row_c[4].isdigit() else None,
                                "grade_level": row_c[5] if len(row_c) > 5 else "",
                                "parent_or_guardian_name": row_c[6] if len(row_c) > 6 else "",
                                "school_name": row_c[8] if len(row_c) > 8 else (row_c[7] if len(row_c) > 7 else ""),
                                "school_location": row_c[9] if len(row_c) > 9 else "Copperbelt",
                                "annual_fees_zmw": clean_amount(row_c[-1]),
                                "academic_year": 2025,
                                "approval_status": "APPROVED"
                            })

df_secondary = pd.DataFrame(sec_records)
df_secondary.drop_duplicates(subset=["constituency", "pupil_name"], inplace=True)
df_secondary.insert(0, "secondary_bursary_id", [f"NCC-CDF-SEC-{i+1:04d}" for i in range(len(df_secondary))])
print(f"Total Secondary Boarding Bursaries Extracted: {len(df_secondary)}")
print(tabulate(df_secondary.head(), headers="keys", tablefmt="psql"))


## 7. Extraction & Cleaning: CDF Empowerment Grants & Loans (2025)

In [ ]:
emp_records = []
grant_files = [
    ("Ndola Central", "LIST-OF-APPROVED-CDF-2025-EMPOWERNMENT-GRANTS-NDOLA-CENTRAL-Constituency-1.pdf", "GRANT"),
    ("Bwana Mkubwa", "LIST-OF-APPROVED-CDF-2025-EMPWERMENT-GRANTS-BWANA-MKUBWA-Constituency-1.pdf", "GRANT"),
    ("Chifubu", "LIST-OF-APPROVED-CDF-2025-EMPOWERMENT-GRANTS-CHIFUBU-CONSTITUENCY-1.pdf", "GRANT"),
    ("Kabushi", "LIST-OF-APROVED-CDF-2025-EMPOWERMENT-GRANTS-KABUSHI-Constituency-1.pdf", "GRANT"),
    ("Ndola Central", "LIST-OF-APPROVED-CDF-2025-EMPOWERMRNT-LOANS-NDOLA-CENTRAL-CINSTITUENCY_1-2.pdf", "LOAN"),
    ("Bwana Mkubwa", "LIST-OF-APPROVED-CDF-2025-LOAN-BENEFICARIES-BWANA-MKUBWA-CONTITUENCY.pdf", "LOAN"),
    ("Chifubu", "LIST-OF-APPROVED-CDF2025-EMPOWERMENT-LOANS-CHIFUBU-CONSTITUENCY-Copy.pdf", "LOAN"),
]

for const_name, fname, mechanism in grant_files:
    fpath = os.path.join(SOURCE_DIR, fname)
    if not os.path.exists(fpath):
        continue
    with pdfplumber.open(fpath) as pdf:
        for page in pdf.pages:
            tables = page.extract_tables()
            if tables:
                for tbl in tables:
                    for row in tbl:
                        row_c = [clean_text(c) for c in row if c is not None]
                        if len(row_c) >= 7 and re.match(r"^\d+$", row_c[0] if row_c[0] != "" else (row_c[1] if len(row_c)>1 else "")):
                            offset = 1 if row_c[0] == "" else 0
                            emp_records.append({
                                "constituency": const_name.upper(),
                                "funding_type": mechanism,
                                "group_or_applicant_name": row_c[offset + 1],
                                "group_category": row_c[offset + 2] if len(row_c) > (offset + 2) else "COMMUNITY",
                                "registration_no": row_c[offset + 3] if len(row_c) > (offset + 3) else "",
                                "ward": row_c[offset + 4] if len(row_c) > (offset + 4) else "",
                                "contact_person": row_c[offset + 5] if len(row_c) > (offset + 5) else "",
                                "business_venture_type": row_c[offset + 7] if len(row_c) > (offset + 7) else "",
                                "economic_sector": row_c[offset + 8] if len(row_c) > (offset + 8) else "AGRICULTURE/TRADE",
                                "amount_approved_zmw": clean_amount(row_c[-1]),
                                "allocation_year": 2025,
                                "approval_status": "APPROVED"
                            })

df_empowerment = pd.DataFrame(emp_records)
df_empowerment.drop_duplicates(subset=["constituency", "group_or_applicant_name", "funding_type"], inplace=True)
df_empowerment.insert(0, "empowerment_id", [f"NCC-CDF-EMP-{i+1:04d}" for i in range(len(df_empowerment))])
print(f"Total CDF Empowerment Grants & Loans Extracted: {len(df_empowerment)}")
print(tabulate(df_empowerment.head(), headers="keys", tablefmt="psql"))


## 8. Extraction & Cleaning: Municipal Budgets & Financial Allocations (2024–2026)

In [ ]:
fin_records = []
budget_2026_path = os.path.join(SOURCE_DIR, "NDOLA-CITY-COUNCIL-2026-BUDGET-1.pdf")
if os.path.exists(budget_2026_path):
    with pdfplumber.open(budget_2026_path) as pdf:
        current_prog = "Local Governance and Institutional Management"
        for page in pdf.pages:
            text = page.extract_text() or ""
            prog_match = re.search(r"Programme\s*:\s*\d+\s*([A-Za-z\s]+)", text)
            if prog_match:
                current_prog = prog_match.group(1).strip()
            tables = page.extract_tables()
            for tbl in tables:
                for row in tbl:
                    row_c = [clean_text(c) for c in row if c is not None]
                    if len(row_c) >= 3:
                        line_desc = row_c[0]
                        if any(k in line_desc.upper() for k in ["PERSONAL EMOLUMENTS", "USE OF GOODS", "TRANSFERS", "CAPITAL", "REVENUE", "LGEF", "RATES", "FEES", "GRANTS", "CDF"]):
                            amt_val = clean_amount(row_c[-1] if len(row_c) > 1 else 0)
                            if amt_val > 0:
                                fin_records.append({
                                    "fiscal_year": 2026,
                                    "programme_name": current_prog,
                                    "line_item_category": line_desc,
                                    "budget_type": "APPROVED_BUDGET",
                                    "amount_zmw": amt_val,
                                    "currency": "ZMW",
                                    "funding_source": "LGEF / Local Revenue / National Grants"
                                })

df_financials = pd.DataFrame(fin_records)
df_financials.drop_duplicates(inplace=True)
df_financials.insert(0, "budget_record_id", [f"NCC-FIN-{i+1:04d}" for i in range(len(df_financials))])
print(f"Total Municipal Budget & Financial Records Extracted: {len(df_financials)}")
print(tabulate(df_financials.head(), headers="keys", tablefmt="psql"))


## 9. Extraction & Cleaning: Council Governance & Committee Resolutions

In [ ]:
gov_records = []
minute_files = [
    ("Full Council Ordinary", "MINUTES-OF-THE-ORDINARY-MEETING-OF-THE-COUNCIL-HELD-ON-1ST-JULY-2025.pdf", "2025-07-01"),
    ("Full Council Special", "MINUTES-OF-THE-SPECIAL-COUNCIL-MEETING-HELD-ON-4TH-APRIL-2025.pdf", "2025-04-04"),
    ("Planning Committee", "MINUTES-OF-PLANNING-HELD-ON-26TH-SEPTEMBER-2024.pdf", "2024-09-26"),
    ("Engineering Services Committee", "MINUTES-OF-ENGINEERING-SERVICES-COMMITTEE-HELD-ON-MONDAY-23RD-SEPTEMBER-2024.pdf", "2024-09-23"),
    ("Audit Committee", "MINUTES-OF-AUDIT-COMMITTEE-HELD-ON-25TH-SEPTEMBER-2024.pdf", "2024-09-25"),
    ("Legal Services Committee", "MINUTES-OF-LEGAL-SERVICES-ADMIN-HELD-ON-1ST-OCTOBER-2024.pdf", "2024-10-01"),
    ("Community Development Committee", "MINUTES-OFCOMMUNITY-DEVELOPMENT-HELD-ON-16TH-SEPTEMBER-2024.pdf", "2024-09-16"),
    ("DDCC Quarterly", "DDCC-MINUTES-OF-2ND-QUARTER-MEETING_22-JULY-2025.pdf", "2025-07-22"),
    ("DDCC Quarterly", "FINAL-DDCC-MINUTES-OF-3RD-QUARTER-MEETING_24-SEPTEMBER-2025.pdf", "2025-09-24"),
]

for comm_name, fname, mdate in minute_files:
    fpath = os.path.join(SOURCE_DIR, fname)
    if not os.path.exists(fpath):
        continue
    with pdfplumber.open(fpath) as pdf:
        full_text = ""
        for p in pdf.pages:
            full_text += (p.extract_text() or "") + "\n"
        items = re.findall(r"(MIN(?:UTE)?\s*[\.\/A-Z0-9\s\/-]+?)(?=\n[A-Z\s]{4,}|\nMIN|\Z)", full_text, re.DOTALL)
        if not items:
            paragraphs = [p.strip() for p in full_text.split("\n\n") if len(p.strip()) > 80 and "RESOLVED" in p]
            for p_idx, p in enumerate(paragraphs[:15]):
                gov_records.append({
                    "committee_or_body": comm_name,
                    "meeting_date": mdate,
                    "agenda_item_or_ref": f"ITEM-{p_idx+1:02d}",
                    "resolution_summary": clean_text(p[:300]),
                    "governance_area": "Municipal Administration & Infrastructure",
                    "status": "ADOPTED"
                })
        else:
            for item_idx, it in enumerate(items[:20]):
                gov_records.append({
                    "committee_or_body": comm_name,
                    "meeting_date": mdate,
                    "agenda_item_or_ref": clean_text(it[:60]),
                    "resolution_summary": clean_text(it[60:400]),
                    "governance_area": "Council Governance & Service Delivery",
                    "status": "ADOPTED"
                })

df_governance = pd.DataFrame(gov_records)
df_governance.drop_duplicates(inplace=True)
df_governance.insert(0, "resolution_id", [f"NCC-RES-{i+1:04d}" for i in range(len(df_governance))])
print(f"Total Governance Records Extracted: {len(df_governance)}")
print(tabulate(df_governance.head(), headers="keys", tablefmt="psql"))


## 10. Pipe-Separated CSV Export & Schema Validation
Strict Output Enforcement:
- **Directory**: `/tabular/`
- **Separator**: `"|"` (Pipe character)
- **Naming Pattern**: `db-unza26-csc4792-[DESCRIPTION].csv`


In [ ]:
export_manifest = [
    ("ndola_council_cdf_community_projects_2024_2025", df_projects),
    ("ndola_council_cdf_skills_bursaries_2025", df_skills),
    ("ndola_council_cdf_secondary_boarding_bursaries_2025", df_secondary),
    ("ndola_council_cdf_empowerment_grants_loans_2025", df_empowerment),
    ("ndola_council_municipal_budget_and_expenditure_2024_2026", df_financials),
    ("ndola_council_governance_and_committee_resolutions_2024_2025", df_governance),
]

summary_export = []

for desc, df in export_manifest:
    fname = f"db-unza26-csc4792-{desc}.csv"
    fpath = os.path.join(TABULAR_DIR, fname)
    df.to_csv(fpath, sep="|", index=False, encoding="utf-8")
    file_size_kb = os.path.getsize(fpath) / 1024
    summary_export.append({
        "Output CSV File": fname,
        "Record Count": len(df),
        "Column Count": len(df.columns),
        "Size (KB)": round(file_size_kb, 2),
        "Separator": "|"
    })

df_summary = pd.DataFrame(summary_export)
print(tabulate(df_summary, headers="keys", tablefmt="psql"))
